# Data Exploration
- This notebook performs exploratory data analysis on the dataset.
- To expand on the analysis, attach this notebook to a cluster with runtime version **16.4.x-cpu-ml-scala2.13**,
edit [the options of pandas-profiling](https://pandas-profiling.ydata.ai/docs/master/rtd/pages/advanced_usage.html), and rerun it.
- Explore completed trials in the [MLflow experiment](#mlflow/experiments/1825839231234310).

In [0]:
%pip install --no-deps ydata-profiling==4.8.3 pandas==2.2.3 visions==0.7.6 tzdata==2024.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 5.2 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 359.5/359.5 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/12.7 MB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━ 6.6/12.7 MB 198.1 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━ 8.6/12.7 MB 86.3 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━ 8.7/12.7 MB 67.6 MB/s eta 0:00:01

...

  Attempting uninstall: ydata-profiling
    Found existing installation: ydata-profiling 4.9.0
    Not uninstalling ydata-profiling at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-3419ba3c-de0f-4b12-9b3a-17fa8c7f9f8e
    Can't uninstall 'ydata-profiling'. No files were found to uninstall.


  Attempting uninstall: visions
    Found existing installation: visions 0.7.5
    Not uninstalling visions at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-3419ba3c-de0f-4b12-9b3a-17fa8c7f9f8e
    Can't uninstall 'visions'. No files were found to uninstall.


  Attempting uninstall: pandas
    Found existing installation: pandas 1.5.3
    Not uninstalling pandas at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-3419ba3c-de0f-4b12-9b3a-17fa8c7f9f8e
    Can't uninstall 'pandas'. No files were found to uninstall.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [0]:
import os
import shutil
import uuid

import mlflow
import pandas as pd

# Download input data from mlflow into a pandas DataFrame
# Create temporary directory to download data
temp_dir = os.path.join(os.environ["SPARK_LOCAL_DIRS"], "tmp", str(uuid.uuid4())[:8])
os.makedirs(temp_dir)

# Download the artifact and read it
training_data_path = mlflow.artifacts.download_artifacts(
    run_id="1550607cc24f4e1996f2c5f920a9f1e8", artifact_path="data", dst_path=temp_dir
)
df = pd.read_parquet(os.path.join(training_data_path, "training_data"))

# Delete the temporary data
shutil.rmtree(temp_dir)

target_col = "target"

# Drop columns created by AutoML and user-specified sample weight column (if applicable) before pandas-profiling
df = df.drop(["_automl_split_col_0000"], axis=1)

Tue Apr 14 14:44:52 2026 Connection to spark from PID  10147
Tue Apr 14 14:44:52 2026 Initialized gateway on port 37463


Tue Apr 14 14:44:52 2026 Connected to spark.


## Semantic Type Detection Alerts

For details about the definition of the semantic types and how to override the detection, see
[Databricks documentation on semantic type detection](https://docs.microsoft.com/azure/databricks/applications/machine-learning/automl#semantic-type-detection).

- Semantic type `categorical` detected for columns `keyword_diversity`, `semi_hsCode`, `sent_price_decouple`. Training notebooks will encode features based on categorical transformations.

## Profiling Results

In [0]:
from ydata_profiling import ProfileReport

df_profile = ProfileReport(
    df, minimal=True, title="Profiling Report", progress_bar=False, infer_dtypes=False
)
profile_html = df_profile.to_html()

displayHTML(profile_html)

Number of variables,50
Number of observations,475
Missing cells,0
Missing cells (%),0.0%
Total size in memory,148.6 KiB
Average record size in memory,320.3 B
Numeric,50
"semi_hsCode has constant value ""8542365157.5""",Constant
"sent_price_decouple has constant value ""1.0""",Constant
keyword_surge_count has 319 (67.2%) zeros,Zeros
keyword_avg_delta_pct has 56 (11.8%) zeros,Zeros
